In [13]:
import os
import csv
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

In [4]:
csv_path = "train.csv"
rows = []
with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

print(f"Wczytano {len(rows)} przykładów")

Wczytano 12672 przykładów


In [6]:
import cv2
import numpy as np

def lbp_hist(img_path, size=(64, 64)):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise ValueError("Nie można wczytać obrazu")

    img = cv2.resize(img, size)

    h, w = img.shape
    lbp = np.zeros((h, w), dtype=np.uint8)

    neighbors = [
        (-1, -1), (-1, 0), (-1, 1),
        (0, 1),
        (1, 1), (1, 0), (1, -1),
        (0, -1)
    ]

    for y in range(1, h - 1):
        for x in range(1, w - 1):
            center = img[y, x]
            code = 0

            for i, (dy, dx) in enumerate(neighbors):
                if img[y + dy, x + dx] >= center:
                    code |= (1 << i)

            lbp[y, x] = code

    hist = np.zeros(256, dtype=np.float32)

    for value in lbp.ravel():
        hist[value] += 1

    hist /= (hist.sum() + 1e-7)

    return hist

In [7]:
X = []
y = []
valid_paths = []
skipped = 0

for row in rows:
    img_path = row["image_path"]
    label = int(row["label"])
    
    if not os.path.exists(img_path):
        print(f"Pominięty nieistniejący: {img_path}")
        skipped += 1
        continue
    
    hist = lbp_hist(img_path)
    X.append(hist)
    y.append(label)
    valid_paths.append(img_path)

X = np.array(X)
y = np.array(y)

print(f"W przygotowaniu: {len(X)} przykładów")
print(f"Pominiono: {skipped}")
print(f"Wymiary X: {X.shape}")

W przygotowaniu: 12672 przykładów
Pominiono: 0
Wymiary X: (12672, 256)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 8870, Test: 3802


In [11]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

train_acc = rf.score(X_train, y_train)
test_acc = rf.score(X_test, y_test)

print("\n" + "="*50)
print("WYNIKI AKURACJI")
print("="*50)
print(f"Akuracja train: {train_acc:.3f}")
print(f"Akuracja test:  {test_acc:.3f}")


WYNIKI AKURACJI
Akuracja train: 1.000
Akuracja test:  0.691


In [17]:
y_pred = rf.predict(X_test)
print("\n" + "="*50)
print("CLASSIFICATION REPORT (test)")
print("="*50)
print(classification_report(y_test, y_pred, zero_division=0))


CLASSIFICATION REPORT (test)
              precision    recall  f1-score   support

           0       0.69      1.00      0.82      2616
           1       0.00      0.00      0.00        59
           2       0.00      0.00      0.00        17
           3       0.00      0.00      0.00        88
           4       0.00      0.00      0.00        57
           5       0.00      0.00      0.00        58
           6       0.50      0.01      0.01       347
           7       1.00      0.02      0.03        59
           8       0.00      0.00      0.00        17
           9       1.00      0.01      0.02        89
          10       0.00      0.00      0.00        60
          11       0.00      0.00      0.00        60
          12       0.37      0.03      0.05       275

    accuracy                           0.69      3802
   macro avg       0.27      0.08      0.07      3802
weighted avg       0.59      0.69      0.57      3802



In [18]:
import joblib
joblib.dump(rf, "chess_rf_model.joblib")
print("Zapisano chess_rf_model.joblib")

Zapisano chess_rf_model.joblib
